<a href="https://colab.research.google.com/github/jandorrvfx-code/tennis_analysis/blob/main/training/tennis_court_keypoints_training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Download dataset using gdown

In [ ]:
# Install gdown for reliable Google Drive downloads
!pip install gdown -q

In [ ]:
# Download the dataset using gdown with the correct file ID
# The file ID '1lhAaeQCmk2y440PmagA0KmIVBIysVMwu' is extracted from the original wget URL.
!gdown 1lhAaeQCmk2y440PmagA0KmIVBIysVMwu -O tennis_court_dataset.zip

In [ ]:
# Unzip the newly downloaded dataset
!unzip -q tennis_court_det_dataset.zip
print('Dataset unzipped successfully.')

# Start code

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms

import json
import cv2
import numpy as np
import random

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


# Create Torch Dataset

In [ ]:
from IPython.core.display import json
class KeypointsDataset(Dataset):
  def __init__(self, img_dir, data_file):
    self.img_dir = img_dir
    with open(data_file, "r") as f:
      self.data = json.load(f)

    self.transforms = transforms.Compose([
        transforms.ToPILImage(),  # PIL image is an object that holds and manipulates an image
        transforms.Resize((224, 224)),
        transforms.ToTensor(),  # Raw numerical representation of an image
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])  # Normalize values so the model understands
    ])

  def __len__(self):
    return len(self.data)

  def __getitem__(self, idx):
    "Return image and keypoints"
    item = self.data[idx]
    img = cv2.imread(f"{self.img_dir}/{item['id']}.png")
    h, w  = img.shape[:2]

    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)  # BGR to RGB
    img = self.transforms(img)  # Apply transforms
    kps = np.array(item['kps']).flatten()  # Convert keypoints from nD to 1D
    kps = kps.astype(np.float32)

    # Adjust keypoints to resized image
    kps[::2] *= 224.0 / w  # Adjust x coordinates
    kps[1::2] *= 224.0 / h  # Adjust y coordinates

    return img, kps




In [ ]:
train_dataset = KeypointsDataset("data/images", "data/data_train.json")  # Train dataset
val_dataset = KeypointsDataset("data/images", "data/data_val.json")  # Validation dataset

train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
var_loader = DataLoader(val_dataset, batch_size=8, shuffle=True)

# Create model

In [ ]:
model = models.resnet50(pretrained=True)
model.fc = torch.nn.Linear(model.fc.in_features, 14*2)  # Replaces the last layer of the neural network

In [ ]:
model = model.to(device)

# Train model

In [ ]:
criterion = torch.nn.MSELoss()  # Define loss function
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)  # lr = learning rate

In [ ]:
num_epochs=20
for epoch in range(num_epochs):
  for i, (imgs, kps) in enumerate(train_loader):
    imgs = imgs.to(device)
    kps = kps.to(device)

    optimizer.zero_grad()  # Flush optimizer gradients
    outputs = model(imgs)  # Have model predict the keypoints on the image
    loss = criterion(outputs, kps)  # Calculate loss
    loss.backward()  # Backward propagation
    optimizer.step()  # Do a learning step

    if i % 10 == 0:  # Log every 10 steps. If i is divisible by 10
      print(f"Epoch {epoch+1}/{num_epochs}, Step {i+1}/{len(train_loader)}, Loss: {loss.item()}")

# Save model

In [ ]:
# Save the model weights
import os

# Ensure the models directory exists
models_dir = '../models'
os.makedirs(models_dir, exist_ok=True)

# Save the model
model_path = os.path.join(models_dir, 'keypoints_model.pth')
torch.save(model.state_dict(), model_path)

print(f"Model saved successfully to: {model_path}")
